# Bank Marketing Intelligence
## Predicting Customer Subscription

**Business problem:** identify customers with a higher likelihood of subscribing to a bank term deposit, using information available before a phone call.

**Dataset:** UCI Bank Marketing, `bank-additional-full.csv`. The target is `y` (yes/no). We exclude `duration` because it is only known after the call and would leak post-call information into a pre-call prediction model.

## 1. Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/bank-additional-full.csv", sep=";")
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.value_counts())

## 2. Data quality and cleaning

In [ ]:
print("Missing cells:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\\nTarget counts:")
display(df["y"].value_counts())
print("\\nTarget percentages:")
display((df["y"].value_counts(normalize=True)*100).round(2))
unknown_counts = {c: int((df[c].astype(str) == "unknown").sum())
                  for c in df.columns if (df[c].astype(str) == "unknown").any()}
display(pd.Series(unknown_counts, name="unknown_count").sort_values(ascending=False))
# Remove exact duplicate records; retain 'unknown' as a meaningful explicit category.
df = df.drop_duplicates().copy()

## 3. Univariate and bivariate EDA
The following cells inspect more than six variables and provide multiple chart types.

In [ ]:
counts = df["y"].value_counts()
plt.figure(figsize=(6,4)); plt.bar(counts.index, counts.values)
plt.title("Target distribution"); plt.ylabel("Records"); plt.show()

plt.figure(figsize=(7,4)); plt.hist(df["age"], bins=30)
plt.title("Age distribution"); plt.xlabel("Age"); plt.ylabel("Frequency"); plt.show()

plt.figure(figsize=(7,4))
plt.boxplot([df.loc[df.y=="no","age"], df.loc[df.y=="yes","age"]], labels=["no","yes"])
plt.title("Age by subscription outcome"); plt.ylabel("Age"); plt.show()

In [ ]:
job_rate = df.groupby("job")["y"].apply(lambda s: (s=="yes").mean()).sort_values(ascending=False)
plt.figure(figsize=(9,4)); plt.bar(job_rate.index, job_rate.values)
plt.xticks(rotation=40, ha="right"); plt.title("Subscription rate by job"); plt.ylabel("Rate"); plt.show()

month_rate = df.groupby("month")["y"].apply(lambda s: (s=="yes").mean())
plt.figure(figsize=(8,4)); plt.plot(month_rate.index, month_rate.values, marker="o")
plt.title("Subscription rate by month"); plt.ylabel("Rate"); plt.show()

corr = df.select_dtypes(include="number").corr()
plt.figure(figsize=(9,6)); plt.imshow(corr, aspect="auto"); plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index); plt.title("Numeric feature correlation"); plt.show()

## 4. Feature engineering and leakage prevention
- `was_previously_contacted`: indicates whether `pdays` is not the sentinel value 999.
- `days_since_previous_contact`: uses 0 for the no-previous-contact case, alongside the indicator.
- `total_campaign_contacts`: combines current and previous campaign contacts.
- `age_group`: groups age into interpretable bands.
- `duration` is excluded because it is measured after the call.

In [ ]:
from src.features import BankFeatureEngineer
X = df.drop(columns=["y", "duration"])
y = (df["y"] == "yes").astype(int)
X_engineered = BankFeatureEngineer().fit_transform(X)
display(X_engineered.head())
print("Engineered shape:", X_engineered.shape)

## 5. Modeling, feature selection, tuning and evaluation
The complete reproducible training workflow is implemented in `train.py`. It:
1. creates a stratified 70/15/15 train/validation/test split;
2. imputes and scales numerical variables and one-hot encodes categorical variables;
3. performs embedded feature selection with ExtraTrees;
4. compares Logistic Regression, Random Forest and Extra Trees;
5. tunes Logistic Regression's C parameter using 3-fold StratifiedKFold GridSearchCV;
6. selects a classification threshold on validation data and evaluates on the held-out test set.

Run the next cell from the project root. The generated test metrics are in `reports/`.

In [ ]:
# Run the full reproducible model comparison and save the trained pipeline
!python train.py
display(pd.read_csv("reports/model_comparison.csv"))

## 6. Interpretation and limitations
Precision answers: among customers predicted as subscribers, what fraction actually subscribed?
Recall answers: among all actual subscribers, what fraction did the model identify?
Validation data is used for threshold selection and model tuning; the held-out test set is reserved for final evaluation.
Historical campaign patterns may not transfer unchanged to a different bank, country or time period. Predictions support prioritization; they do not guarantee outcomes.